In [ ]:
import pandas as pd
import numpy as np
import os

# 분석에 사용할 원본 CSV 파일 경로를 지정
file_cctv = "../data/대구광역시 북구_데이터통합플랫폼_CCTV위치_20210630.csv"
file_complaint = "../data/대구광역시 북구_불법주정차 민원_20211123.csv"

# CCTV 파일은 UTF-8(BOM 포함) 인코딩이라 utf-8-sig로 불러옵니다.
cctv = pd.read_csv(file_cctv, encoding="utf-8-sig")

# 데이터가 정상적으로 불러와졌는지 확인합니다.
print(cctv.head())

# 컬럼명, 데이터 크기, 결측치를 확인합니다.
print("컬럼:", cctv.columns.tolist())
print("데이터 크기:", cctv.shape)
print(cctv.isnull().sum())

# 설치목적별 대수와 데이터기준일자를 함께 확인합니다.
# 주정차단속 CCTV는 2020-10-26 기준, 나머지는 2021-06-30 기준으로 조사된 자료입니다.
print(cctv.groupby(["데이터기준일자", "설치목적구분"]).size())

# 위도/경도 범위를 확인합니다. (대구: 위도 약 35, 경도 약 128)
print("위도 범위:", cctv["위도"].min(), "~", cctv["위도"].max())
print("경도 범위:", cctv["경도"].min(), "~", cctv["경도"].max())

# 같은 좌표에 CCTV가 여러 대 있는지 확인합니다.
# 한 기둥에 여러 방향 카메라를 다는 경우가 있어서 삭제하지 않고 '대수'로 셉니다.
same_place = cctv.duplicated(subset=["위도", "경도"]).sum()
print("CCTV 대수:", len(cctv))
print("설치 장소 수(좌표 기준):", len(cctv) - same_place)

# 지번주소가 있으면 지번주소를, 없으면 도로명주소를 대표 주소로 사용합니다.
cctv["주소"] = cctv["소재지지번주소"].fillna(cctv["소재지도로명주소"])

# 지번주소에서 동 이름을 뽑습니다. (칠성동2가 -> 칠성동, 산격3동 -> 산격동)
dong = cctv["소재지지번주소"].str.extract(r"([가-힣]+?)\d*(동)(?:\d가)?")
cctv["동"] = dong[0] + dong[1]

print("지번주소로 동을 찾은 CCTV:", cctv["동"].notnull().sum())
print("동을 못 찾은 CCTV:", cctv["동"].isnull().sum())

In [ ]:
#도로명에 동로가 들어간 CCTV가 26대있음. 
#전부 지번주소가 비어 있는상태입니다
#이 주소에서 동을 뽑으면 이렇게 잘못 나옵니다
#신천동로148길 52 → 신천동 (북구에 없는 동)
#대동로1길 43 → 대동 (북구에 없는 동)
#학정동로87 → 학정동 (우연히 맞긴 하지만 도로 이름이라 믿을 수 없어요)
#이경우 도로명으로 동을 뽑지 않고, 좌표로 가장 가까운 민원 지점의 동을 붙입니다. 
#예를 들어 신천동로148길 52는 좌표상 가장 가까운 민원 지점이 있는 동으로 들어갑니다.

# 민원 파일을 불러와 주소별 동과 좌표를 준비합니다.
complaint = pd.read_csv(file_complaint, encoding="cp949")

# 민원 파일은 경도/위도 컬럼 이름이 서로 바뀌어 있어서 바로잡습니다.
complaint = complaint.rename(columns={"경도": "위도", "위도": "경도",
                                      "민원발생누적건수": "민원건수", "민원발생년도": "연도"})

dong = complaint["주소"].str.extract(r"([가-힣]+?)\d*(동)(?:\d가)?")
complaint["동"] = dong[0] + dong[1]

complaint_place = complaint.dropna(subset=["동"]).drop_duplicates("주소")
print("동을 알 수 있는 민원 지점 수:", len(complaint_place))

# CCTV 좌표와 가장 가까운 민원 지점의 동을 찾는 함수입니다.
def nearest_dong(lat, lon):
    distance = (complaint_place["위도"] - lat) ** 2 + (complaint_place["경도"] - lon) ** 2
    return complaint_place.loc[distance.idxmin(), "동"]

# 동이 비어 있는 CCTV만 좌표로 채웁니다.
empty = cctv["동"].isnull()
cctv.loc[empty, "동"] = cctv[empty].apply(lambda row: nearest_dong(row["위도"], row["경도"]), axis=1)

print("동을 못 찾은 CCTV:", cctv["동"].isnull().sum())
print(cctv["동"].value_counts())

# 최종 전처리 결과를 확인합니다.
print("최종 데이터 크기:", cctv.shape)
display(cctv[["주소", "동", "설치목적구분", "위도", "경도", "데이터기준일자"]].head())

# 전처리 결과를 저장합니다.
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/lee_cctv_location.csv"
cctv.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

In [ ]:
# 그래프를 그리기 위한 matplotlib를 불러옵니다.
import matplotlib.pyplot as plt

# 윈도우에서 그래프 한글이 깨지지 않도록 맑은 고딕을 사용합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 그래프를 저장할 폴더가 없으면 만듭니다.
os.makedirs("images", exist_ok=True)

# 설치목적마다 같은 색을 쓰도록 정해 둡니다.
purpose_order = ["주정차단속", "교통단속", "어린이보호"]
purpose_color = {"주정차단속": "tab:red", "교통단속": "tab:blue", "어린이보호": "tab:green"}

In [ ]:
# 설치목적별 CCTV 대수를 계산합니다.
purpose_count = cctv["설치목적구분"].value_counts().reindex(purpose_order)
print(purpose_count)
print()
print((purpose_count / len(cctv) * 100).round(1))

# 설치목적별 대수를 막대그래프로 비교합니다.
plt.figure(figsize=(7, 5))
plt.bar(purpose_count.index, purpose_count.values,
        color=[purpose_color[p] for p in purpose_count.index])
plt.title("설치목적별 CCTV 대수 (대구 북구)")
plt.xlabel("설치목적")
plt.ylabel("CCTV 대수")
plt.tight_layout()
plt.savefig("images/lee_cctv_purpose_count.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 설치목적별로 지도를 나눠서 그립니다. (다른 목적 CCTV는 회색으로 깔아 둡니다)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

for ax, purpose in zip(axes, purpose_order):
    part = cctv[cctv["설치목적구분"] == purpose]
    ax.scatter(cctv["경도"], cctv["위도"], s=6, c="lightgray")
    ax.scatter(part["경도"], part["위도"], s=18, c=purpose_color[purpose], alpha=0.8)
    ax.set_title(f"{purpose} CCTV ({len(part)}대)")
    ax.set_xlabel("경도")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("위도")
plt.suptitle("설치목적별 CCTV 위치 비교", fontsize=15)
plt.tight_layout()
plt.savefig("images/lee_cctv_map_by_purpose.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 동 x 설치목적별 CCTV 대수를 교차 집계합니다.
dong_purpose = pd.crosstab(cctv["동"], cctv["설치목적구분"]).reindex(columns=purpose_order, fill_value=0)
dong_purpose["합계"] = dong_purpose.sum(axis=1)
dong_purpose = dong_purpose.sort_values("합계", ascending=False)

display(dong_purpose)

# 동별 CCTV 대수를 설치목적별 누적 막대그래프로 나타냅니다.
dong_purpose.drop(columns="합계").plot(
    kind="bar", stacked=True, figsize=(12, 6),
    color=[purpose_color[p] for p in purpose_order]
)
plt.title("동별 CCTV 설치 대수 (설치목적별)")
plt.xlabel("동")
plt.ylabel("CCTV 대수")
plt.xticks(rotation=45)
plt.legend(title="설치목적")
plt.tight_layout()
plt.savefig("images/lee_cctv_dong_purpose.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CCTV가 많이 모여 있는 곳을 보기 위해 육각형 칸(hexbin)으로 밀집도를 나타냅니다.
plt.figure(figsize=(9, 8))
plt.hexbin(cctv["경도"], cctv["위도"], gridsize=25, cmap="Blues", mincnt=1)
plt.colorbar(label="칸 안의 CCTV 대수")
plt.title("대구 북구 CCTV 밀집도")
plt.xlabel("경도")
plt.ylabel("위도")
plt.tight_layout()
plt.savefig("images/lee_cctv_density.png", dpi=150, bbox_inches="tight")
plt.show()